# 한국 기업 분기별 매출 시계열 예측 — `Korea_revenue_forecast_v7` (지속 사용)

DataGuide 원천(`korea_fs_data_from_DG`)에서 분기 매출액(`item_code='M000904001'`)을 조회해
**SARIMA · ETS · Theta + 앙상블**로 예측하고, 결과를 `korea_revenue_forecast_result`에 저장합니다.

### v6 → v7 변경사항

| # | 항목 | 내용 |
|---|---|---|
| 1 | **매출 조회 강건화** | `item_code` 단독 필터 → **`item_code` OR `indicator`('매출액') 이중 매칭**. 최근 적재분이 item_code 없이 indicator 만으로 들어온 경우(예: A003350 의 2025Q4~2026Q2) 예측 앵커가 과거 분기로 밀리던 문제 해결. 분기 중복 시 item_code 매칭 우선. indicator 로 보완된 분기는 ⚠ 경고로 표시 (ETL item_code 백필 권장) |
| 2 | **앵커 로그** | verbose 시 `[anchor] 실적 마지막 분기 → 예측 시작~종료 분기` 를 출력해 stale 앵커를 즉시 확인 가능 |

### v5 → v6 변경사항

| # | 항목 | 내용 |
|---|---|---|
| 1 | **1회성 코드 분리** | 마이그레이션(§6)을 `Korea_revenue_forecast_DB_migration_v1.ipynb` 로 이관. 이 노트북은 **위에서 아래로 전부 실행**하는 정기 예측 전용 |
| 2 | **표준은 그대로 내장** | vintage 키 자동 보증(`ensure_vintage_key`) · 저장 티커 표준화(`RESULT_TICKER_HAS_A_PREFIX`) · 단위 표준(`RESULT_VALUE_UNIT`)은 저장 경로에 남아 매 실행 시 적용 |

> **첫 정기 실행 전 1회**: 마이그레이션 노트북을 먼저 실행해 기존 테이블을 정리하세요.
> 그 이후로는 이 노트북만 사용하면 됩니다. 새 DB·테이블은 만들어지지 않으며,
> 기존 `investar.korea_revenue_forecast_result` 에 계속 저장됩니다.
> ⚠️ `RESULT_TICKER_HAS_A_PREFIX` 는 마이그레이션 노트북과 같은 값이어야 합니다 (기본 `False`).

### v4 → v5 변경사항 (요약)

| # | 항목 | 내용 |
|---|---|---|
| 1 | **vintage 키 자동 보증** | 저장 시 실제 DB 키를 `information_schema` 로 검사, 구버전 3열 키면 자동 `ALTER` → 과거 예측 이력 보존 |
| 2 | **저장 티커 표준화** | `RESULT_TICKER_HAS_A_PREFIX` 로 단일화 (기본 `False` = '005930') |
| 3 | **저장 단위 표준화** | `RESULT_VALUE_UNIT` (기본 '천원' = DG 원천 단위) |

### v3 → v4 변경사항 (요약)

결측 분기 연속구간 검증 · 전 모형 실패 차단 · 앙상블 집계 선택(`ENSEMBLE_AGG`) · `n_models` 기록 등.
상세는 v4 노트북 헤더 참조.

### 노트북 구성
1. **환경 설정** — import · 경로 · 모듈 · DB
2. **데이터 조회 / 예측 / 저장 함수**
3. **개별 종목 점검** `inspect_ticker()` — DB 저장 없이 타당성 확인
4. **일괄 예측 실행** — `BATCH_SIZE`(기본 50)개씩 모아 저장, `GC_EVERY`개마다 `gc.collect()`


## 1. 환경 설정

In [1]:
import sys
import os
import gc
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime
from pathlib import Path
from typing import Union, List, Tuple, Dict, Optional
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

### 1-1. 데이터 소스 설정
필요 시 이 셀의 상수만 수정하면 됩니다.

In [2]:
# ==================== 데이터 소스 설정 (필요 시 여기만 수정) ====================
# 매출 데이터를 조회할 원천 테이블
FS_TABLE = "korea_fs_data_from_DG"

# DataGuide에서 매출액을 가리키는 item_code
#   M000904001 = '매출액(천원)' (IS 시트 원본)
REVENUE_ITEM_CODE = "M000904001"

# ticker 형식: True면 'A005930' (A 접두사 포함), False면 '005930'
TICKER_HAS_A_PREFIX = True

# ==================== v7: 매출 조회 강건화 ====================
# 배경: 최근 적재분(예: A003350 의 2025Q4~2026Q2)이 item_code 없이(또는 다른 코드로)
#       indicator('매출액')만으로 적재되면, item_code 단독 필터는 최근 분기를 놓쳐
#       예측 앵커가 과거 분기로 밀린다 (→ 이미 실현된 분기를 "예측"으로 저장).
# 처리: (item_code = REVENUE_ITEM_CODE) OR (TRIM(indicator) IN 후보) 이중 매칭.
#       같은 분기에 둘 다 있으면 item_code 매칭 행 우선.
USE_INDICATOR_FALLBACK = True
REVENUE_INDICATOR_CANDIDATES = ["매출액", "매출액(천원)"]   # TRIM 후 정확 일치


def normalize_ticker_for_dg(ticker) -> str:
    """
    DG 테이블 조회용 ticker 정규화.
    입력이 '005930', 5930, 'A005930' 어떤 형태든
    DG 스키마에 맞게 'A005930' 형태로 통일한다.
    """
    if isinstance(ticker, int):
        raw = f"{ticker:06d}"
    else:
        raw = str(ticker).strip()

    # 'A' 접두사 제거 (이미 있으면) 후 6자리 zfill
    body = raw[1:] if raw.upper().startswith('A') else raw
    body = body.zfill(6)

    return f"A{body}" if TICKER_HAS_A_PREFIX else body


# ==================== 결과 테이블 저장 표준 (v5) ====================
RESULT_TABLE = "korea_revenue_forecast_result"

# 결과 테이블에 저장하는 ticker 형식.
#   False : '005930'  (DART · pykrx · KRX 계열과 조인 호환 — 기본값)
#   True  : 'A005930' (DataGuide 원본 형식)
# 이 값이 곧 테이블의 표준이므로, 소비하는 쪽(PSR 밸류에이션 등) 형식에 맞춰 한 번만 정한다.
RESULT_TICKER_HAS_A_PREFIX = False

# 값 단위. 원천 M000904001은 '천원'.
SOURCE_VALUE_UNIT = "천원"
RESULT_VALUE_UNIT = "천원"          # '천원' | '원' | '백만원'

_UNIT_IN_WON = {"원": 1, "천원": 1_000, "백만원": 1_000_000}
# 소스 단위 -> 결과 단위 환산 계수 (기본 천원->천원 = 1.0)
RESULT_UNIT_FACTOR = _UNIT_IN_WON[SOURCE_VALUE_UNIT] / _UNIT_IN_WON[RESULT_VALUE_UNIT]


def normalize_ticker_for_result(ticker) -> str:
    """결과 테이블 저장용 ticker 정규화 (RESULT_TICKER_HAS_A_PREFIX 기준)."""
    if isinstance(ticker, int):
        raw = f"{ticker:06d}"
    else:
        raw = str(ticker).strip()
    body = raw[1:] if raw.upper().startswith("A") else raw
    body = body.zfill(6)
    return f"A{body}" if RESULT_TICKER_HAS_A_PREFIX else body


### 1-2. 경로 설정
`DATA` 폴더를 자동 탐색해 `sys.path`에 추가합니다 (랩탑/데스크탑 등 PC 무관).

In [3]:
def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정 v2 — DATA 폴더 자동 탐색.
    탐색 순서:
      ① 환경변수 DATA_DIR (직접 지정 시)
      ② 현재 폴더와 상위 폴더들의 직속 DATA
      ③ 현재 폴더 하위(깊이 3) 및 상위 3단계 폴더들의 하위(깊이 2)
    old_files/backup 등 백업 폴더 내의 DATA 는 무시한다.
    이 노트북은 대만 프로젝트(config.py)와 무관하며 DATA 폴더만 있으면 동작한다.
    """
    import os
    SKIP = {".git", ".idea", ".venv", "venv", "env", "__pycache__",
            "node_modules", "site-packages", ".ipynb_checkpoints",
            "AppData", "anaconda3", "Miniconda3",
            "old_files", "old", "_old", "backup", "backups", "archive", "bak"}
    current = Path.cwd()

    def register(data_folder):
        parent = data_folder.parent
        for p in (str(parent), str(data_folder)):
            if p not in sys.path:
                sys.path.insert(0, p)
        print("=" * 80)
        print("경로 설정 완료")
        print("=" * 80)
        print(f"프로젝트 루트: {parent}")
        print(f"DATA 폴더:    {data_folder}")
        print(f"현재 위치:     {current}")
        print("=" * 80 + "\n")
        return {"project_root": parent, "data_folder": data_folder, "current": current}

    # ① 환경변수
    env = os.environ.get("DATA_DIR")
    if env and Path(env).exists():
        return register(Path(env))

    # ② 상위 방향: 직속 DATA
    for parent in [current, *current.parents]:
        d = parent / "DATA"
        if d.exists():
            return register(d)

    # ③ 하위 방향: cwd 깊이 3, 상위 3단계는 깊이 2
    def scan(root, max_depth):
        root = Path(root); base = len(root.parts)
        for dp, dn, _ in os.walk(root):
            if len(Path(dp).parts) - base >= max_depth:
                dn[:] = []
            dn[:] = [x for x in dn if x not in SKIP and not x.startswith(".")]
            if "DATA" in dn:
                return Path(dp) / "DATA"
        return None
    hit = scan(current, 3)
    if hit is None:
        for parent in list(current.parents)[:3]:
            hit = scan(parent, 2)
            if hit:
                break
    if hit:
        return register(hit)

    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}\n"
        "아래처럼 직접 지정한 뒤 이 셀을 다시 실행하세요:\n"
        '  import os; os.environ["DATA_DIR"] = r"C:\\...\\DATA"'
    )


# 경로 설정 실행
paths = setup_universal_paths()

경로 설정 완료
프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측



### 1-3. 예측 모듈 import

> **v4 변경:** `universal_ts_forecast_function_v2` 에서 import 합니다.
> DATA 폴더의 실제 파일명과 일치해야 하며, 다르면 아래 `MODULE_NAME` 만 바꾸면 됩니다.


In [4]:
# 예측 모듈명 (DATA 폴더의 실제 파일명과 일치해야 함)
MODULE_NAME = "universal_ts_forecast_function_v3"

import importlib

try:
    _fc_mod = importlib.import_module(MODULE_NAME)
except ModuleNotFoundError:
    # 구버전 파일명으로 폴백 (환경별 파일명 차이 대응)
    _fc_mod = importlib.import_module("universal_ts_forecast_function")
    MODULE_NAME = "universal_ts_forecast_function"
    print("[경고] _v2 모듈을 찾지 못해 구버전으로 폴백했습니다.")

forecast_sarima = _fc_mod.forecast_sarima
forecast_ets    = _fc_mod.forecast_ets
forecast_theta  = _fc_mod.forecast_theta

from stock_invest_function import get_db_host

print(f"예측 모듈 import 성공  ({MODULE_NAME})")
print(f"  파일 위치: {getattr(_fc_mod, '__file__', 'n/a')}")


예측 모듈 import 성공  (universal_ts_forecast_function_v3)
  파일 위치: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\universal_ts_forecast_function_v3.py


### 1-4. DB 연결 설정

In [5]:
def get_db_info() -> dict:
    """DB 연결 정보 반환"""
    return {
        "host": get_db_host(),
        "port": 3307,
        "user": "stox7412",
        "password": "Apt106503!~",
        "database": "investar",
    }


def get_connection(db_info: dict):
    """DB 연결 생성 (pymysql)"""
    return pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4",
    )

## 2. 데이터 조회 함수

In [6]:
def get_all_tickers(db_info: dict) -> List[str]:
    """
    DataGuide 테이블에서 매출 데이터가 존재하는 모든 ticker 조회.
    매출액(item_code) 레코드가 하나라도 있는 기업만 대상.
    """
    conn = get_connection(db_info)
    try:
        if USE_INDICATOR_FALLBACK and REVENUE_INDICATOR_CANDIDATES:
            ind_ph = ", ".join(["%s"] * len(REVENUE_INDICATOR_CANDIDATES))
            sql = f"""
                  SELECT DISTINCT ticker
                  FROM {FS_TABLE}
                  WHERE TRIM(item_code) = %s OR TRIM(indicator) IN ({ind_ph})
                  ORDER BY ticker
                  """
            params = [REVENUE_ITEM_CODE] + list(REVENUE_INDICATOR_CANDIDATES)
        else:
            sql = f"""
                  SELECT DISTINCT ticker
                  FROM {FS_TABLE}
                  WHERE item_code = %s
                  ORDER BY ticker
                  """
            params = [REVENUE_ITEM_CODE]
        df = pd.read_sql(sql, conn, params=params)
        tickers = df['ticker'].tolist()
        print(f"매출 데이터 보유 ticker {len(tickers)}개 조회 완료 (소스: {FS_TABLE})\n")
        return tickers
    finally:
        conn.close()


def get_quarterly_revenue_simple(
        db_info: dict,
        ticker: Union[str, int],
        adjust_q4: bool = False,
        verbose: bool = False
) -> pd.DataFrame:
    """
    특정 ticker의 분기별 매출 시계열을 DataGuide 테이블에서 조회.
    DataGuide는 각 분기가 독립값이므로 FY 역산이 불필요하다.

    Returns columns:
        report_date, thstrm_amount, bsns_year, quarter, ticker
    """
    ticker_str = normalize_ticker_for_dg(ticker)
    conn = get_connection(db_info)
    try:
        # ── v7: item_code OR indicator 이중 매칭 ─────────────────
        if USE_INDICATOR_FALLBACK and REVENUE_INDICATOR_CANDIDATES:
            ind_ph = ", ".join(["%s"] * len(REVENUE_INDICATOR_CANDIDATES))
            sql = f"""
                  SELECT
                      date                  AS report_date,
                      value                 AS thstrm_amount,
                      YEAR(date)            AS bsns_year,
                      ticker                AS ticker,
                      CASE WHEN TRIM(item_code) = %s THEN 1 ELSE 0 END AS by_item_code
                  FROM {FS_TABLE}
                  WHERE ticker = %s
                    AND (TRIM(item_code) = %s OR TRIM(indicator) IN ({ind_ph}))
                    AND value IS NOT NULL
                  ORDER BY date
                  """
            params = ([REVENUE_ITEM_CODE, ticker_str, REVENUE_ITEM_CODE]
                      + list(REVENUE_INDICATOR_CANDIDATES))
        else:
            sql = f"""
                  SELECT
                      date                  AS report_date,
                      value                 AS thstrm_amount,
                      YEAR(date)            AS bsns_year,
                      ticker                AS ticker,
                      1                     AS by_item_code
                  FROM {FS_TABLE}
                  WHERE ticker = %s
                    AND item_code = %s
                    AND value IS NOT NULL
                  ORDER BY date
                  """
            params = [ticker_str, REVENUE_ITEM_CODE]
        df = pd.read_sql(sql, conn, params=params)

        if df.empty:
            if verbose:
                print(f"  {ticker_str}: 데이터 없음")
            return df

        # quarter 컬럼 유도 (downstream 호환)
        df['quarter'] = 'Q' + ((pd.to_datetime(df['report_date']).dt.month + 2) // 3).astype(str)

        # ── v7: 분기 중복 해소 — item_code 매칭 우선, 그다음 최신 date ──
        df['_per'] = pd.to_datetime(df['report_date']).dt.to_period('Q')
        df = (df.sort_values(['_per', 'by_item_code', 'report_date'])
                .drop_duplicates(subset=['_per'], keep='last')
                .reset_index(drop=True))

        # ── v7: 커버리지 진단 — indicator 로만 확보된 최근 분기 경고 ──
        ic_pers  = df.loc[df['by_item_code'] == 1, '_per']
        ind_pers = df.loc[df['by_item_code'] == 0, '_per']
        ic_last  = ic_pers.max() if len(ic_pers) else None
        filled = sorted(str(p) for p in ind_pers
                        if (ic_last is None or p > ic_last))
        if filled:
            print(f"  [경고] {ticker_str}: item_code({REVENUE_ITEM_CODE}) 미적재 "
                  f"최근 분기 {len(filled)}개를 indicator 매칭으로 보완 {filled} "
                  f"(item_code 마지막={ic_last}) — 적재 파이프라인에서 item_code 백필 권장")

        if verbose:
            print(f"  {ticker_str}: 매출 {len(df)}분기 "
                  f"(item_code {int((df['by_item_code'] == 1).sum())} · "
                  f"indicator 보완 {int((df['by_item_code'] == 0).sum())})  "
                  f"기간 {df['_per'].min()} ~ {df['_per'].max()}")

        df = df.drop(columns=['_per', 'by_item_code'])

        if adjust_q4 and verbose:
            print(f"  [INFO] DG 데이터는 분기 독립값이므로 adjust_fy_to_q4 skip")

        return df
    finally:
        conn.close()

<details>
<summary>📦 [LEGACY] <code>adjust_fy_to_q4()</code> — DART 데이터용 (현재 미사용, 보존용)</summary>

DataGuide 경로에서는 호출하지 않습니다. 과거 DART(누적 공시) 경로로 돌아갈 경우를 대비해 보존합니다.
</details>

In [7]:
def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """
    [LEGACY] DART 데이터용. DataGuide 경로에서는 사용하지 않음.
    각 연도별로 FY - (Q1 + Q2 + Q3) = 순수 Q4 로 역산.
    """
    result_df = df.copy()
    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        fy_mask = year_mask & (result_df['quarter'] == 'FY')
        if fy_mask.any():
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'H1', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()
            q4_amount = fy_amount - q123_sum
            result_df.loc[fy_mask, 'thstrm_amount'] = q4_amount
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
    return result_df

## 3. 예측 함수

`_prepare_series()`로 매출 시계열(PeriodIndex)을 만든 뒤 세 모델을 적합합니다.
점검용 `inspect_ticker()`와 일괄용 `forecast_quarterly_revenue()`가 **동일한 전처리·모델 호출**을
공유하도록 분리해, 점검 시 보이는 예측값과 실제 저장값이 어긋나지 않게 했습니다.

> **v4:** `_prepare_series()`에 **최근 연속 구간 추출**이 추가되었습니다.
> 중간에 빠진 분기가 있으면 마지막 결측 이후 구간만 사용합니다.
> `_run_models()`는 **전 모형 실패 시 예외**를 던져 NULL 저장을 차단합니다.


In [8]:
# ==================== 앙상블 집계 방식 ====================
# 'median' : 한 모형이 크게 튈 때 방어됨 (배치 권장, v4 기본값)
# 'mean'   : v3 동작과 동일
ENSEMBLE_AGG = "median"


def _prepare_series(revenue_df: pd.DataFrame, verbose: bool = False) -> pd.Series:
    """
    조회된 매출 DataFrame을 분기 PeriodIndex 매출 시계열(float)로 변환.

    v4 변경 — 최근 연속 구간만 사용
        이 노트북은 v2의 forecast_one_from_pivot_inline() 이 아니라 forecast_* 를 직접
        호출하므로 v2 내부의 extract_recent_contiguous_block()(연속구간 검증)을 거치지
        않는다. 중간에 결측 분기가 있으면 그대로 이어붙여 '연속'으로 간주해 적합되므로
        조용히 잘못된 예측이 만들어진다. 여기서 직접 차단한다.

    중복 분기는 마지막 값 사용.
    """
    df = revenue_df[['report_date', 'thstrm_amount']].copy()
    df.columns = ['date', 'revenue']
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['quarter'] = df['date'].dt.quarter
    df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
    df['period'] = pd.PeriodIndex(df['period'], freq='Q')

    # 중복 분기 제거 (각 분기의 마지막 값)
    if df.duplicated(subset=['period'], keep=False).any():
        df = df.sort_values(['period', 'date']).groupby('period').last().reset_index()

    df = df.set_index('period').sort_index()
    s = df['revenue'].astype(float).dropna()

    if s.empty:
        return s

    # ---- 최근 연속 구간 추출 (v4 신규) ----
    full = pd.period_range(s.index.min(), s.index.max(), freq='Q')
    if len(full) != len(s):
        missing = full.difference(s.index)
        n_before = len(s)
        s = s.loc[s.index > missing.max()]
        if verbose:
            print(f"  [결측분기 {len(missing)}개] {n_before}분기 -> 최근 연속 {len(s)}분기만 사용 "
                  f"(마지막 결측: {missing.max()})")

    return s


def _run_models(series: pd.Series, forecast_quarters: int) -> Tuple[pd.DataFrame, dict]:
    """
    SARIMA / ETS / Theta 적합 후 예측 분기 인덱스를 가진 wide DataFrame과
    각 모델의 원본 result dict를 반환.

    v4 변경
      - 전 모형 실패 시 ValueError (v3는 NULL 행을 그대로 DB에 저장했음)
      - 앙상블 집계를 ENSEMBLE_AGG 로 선택 (기본 median)
      - 성공 모형 수를 n_models 컬럼으로 기록
    """
    m = 4  # 분기 데이터의 계절성

    sarima_result = forecast_sarima(
        y=series, forecast_horizon=forecast_quarters,
        seasonal_period=m, try_transforms=True,
    )
    ets_result = forecast_ets(
        y=series, forecast_horizon=forecast_quarters,
        m=m, try_transforms=True,
    )
    theta_result = forecast_theta(
        y=series, forecast_horizon=forecast_quarters,
        m=m, try_transforms=True,
    )

    raw = {"SARIMA": sarima_result, "ETS": ets_result, "Theta": theta_result}

    # ---- 전 모형 실패 차단 (v4 신규) ----
    ok_models = [k for k, r in raw.items() if isinstance(r, dict) and "forecast" in r]
    if not ok_models:
        errs = "; ".join(f"{k}:{(r or {}).get('error', 'unknown')}" for k, r in raw.items())
        raise ValueError(f"전 모형 적합 실패 ({errs})")

    last_period = series.index[-1]
    forecast_periods = pd.period_range(start=last_period + 1, periods=forecast_quarters, freq='Q')

    fc = pd.DataFrame(index=forecast_periods)
    for name in ("SARIMA", "ETS", "Theta"):
        vals = raw[name].get("forecast") if isinstance(raw[name], dict) else None
        # 실패 모형은 NaN 컬럼으로 채워 dtype을 float으로 유지
        fc[name] = np.asarray(vals, dtype=float) if vals is not None else np.nan

    cols = ["SARIMA", "ETS", "Theta"]
    fc["Ensemble"] = (fc[cols].median(axis=1) if ENSEMBLE_AGG == "median"
                      else fc[cols].mean(axis=1))
    fc["n_models"] = fc[cols].notna().sum(axis=1)

    return fc, raw


def forecast_quarterly_revenue(
        ticker: str,
        db_info: dict,
        forecast_quarters: int = 8,
        min_data_periods: int = 24,
        verbose: bool = False
) -> Tuple[bool, pd.DataFrame, str]:
    """
    단일 ticker 분기 매출 예측.
    Returns: (성공여부, wide 예측 DataFrame[date,SARIMA,ETS,Theta,Ensemble,n_models,ticker], 에러메시지)

    v4 변경
      - 길이 검증을 '연속 구간 추출 이후'에 수행 (결측으로 실제 사용 가능 구간이
        짧아진 경우를 잡아낸다)
      - to_timestamp(how='end') 에 .normalize() 추가
    """
    try:
        revenue_df = get_quarterly_revenue_simple(db_info, ticker, adjust_q4=False, verbose=verbose)
        if revenue_df.empty:
            return False, pd.DataFrame(), "데이터 없음"

        series = _prepare_series(revenue_df, verbose=verbose)

        # 연속구간 추출 후 길이 검증 (v4: 순서 변경)
        if len(series) < min_data_periods:
            return False, pd.DataFrame(), f"데이터 부족 ({len(series)}개 < {min_data_periods}개)"

        # ── v7: 앵커 로그 — stale 입력 즉시 확인용 ──
        if verbose:
            _last = series.index[-1]
            print(f"  [anchor] {normalize_ticker_for_dg(ticker)}: 실적 마지막 {_last} "
                  f"→ 예측 {_last + 1} ~ {_last + forecast_quarters}")

        fc, _ = _run_models(series, forecast_quarters)

        # ---- v5: 저장 표준 적용 (단위 환산 · 티커 정규화) ----
        if RESULT_UNIT_FACTOR != 1:
            for _c in ("SARIMA", "ETS", "Theta", "Ensemble"):
                fc[_c] = fc[_c] * RESULT_UNIT_FACTOR
        fc['ticker'] = normalize_ticker_for_result(ticker)
        # DatetimeIndex로 변환 (분기 말일, 시각 제거)
        fc.index = fc.index.to_timestamp(how='end').normalize()
        fc = fc.reset_index()
        fc.columns = ['date'] + list(fc.columns[1:])
        return True, fc, ""

    except Exception as e:
        error_msg = str(e)
        if verbose:
            print(f"  {ticker}: 예측 실패 - {error_msg}")
        return False, pd.DataFrame(), error_msg


## 4. 🔍 개별 종목 점검 — `inspect_ticker()`  *(요구사항 3)*

종목 코드 하나를 넣으면 **예측 입력으로 쓰인 매출 `tail(20)`** 과 **모델별 예측값**(SARIMA·ETS·Theta·Ensemble)을
표로 보여줍니다. **DB에 저장하지 않으므로** 예측 타당성을 자유롭게 확인할 수 있습니다.

반환값(dict)으로 `input_series`, `input_tail`, `forecast`, `raw_results`를 받아 추가 분석도 가능합니다.

In [9]:
def inspect_ticker(
        ticker: Union[str, int],
        db_info: dict = None,
        forecast_quarters: int = 8,
        tail_n: int = 20,
        show: bool = True
) -> dict:
    """
    개별 종목의 예측 입력값과 모델별 예측값을 조회 (타당성 검증용, DB 저장 X).

    Parameters
    ----------
    ticker : 종목 코드 ('005930', 5930, 'A005930' 모두 허용)
    db_info : DB 연결 정보 (None이면 get_db_info() 자동 호출)
    forecast_quarters : 예측 분기 수
    tail_n : 입력 매출 시계열에서 보여줄 최근 분기 수 (기본 20)
    show : True면 화면에 출력

    Returns
    -------
    dict: {ticker, n_obs, input_series, input_tail, forecast, raw_results}
    """
    if db_info is None:
        db_info = get_db_info()

    ticker_str = normalize_ticker_for_dg(ticker)
    revenue_df = get_quarterly_revenue_simple(db_info, ticker, adjust_q4=False, verbose=False)

    if revenue_df.empty:
        print(f"[{ticker_str}] 매출 데이터가 없습니다.")
        return {"ticker": ticker_str, "n_obs": 0, "input_series": None,
                "input_tail": None, "forecast": None, "raw_results": None}

    series = _prepare_series(revenue_df, verbose=True)
    n_obs = len(series)
    n_raw = len(revenue_df)

    # 입력 매출 tail(N) — 보기 좋게 정리
    tail = series.tail(tail_n).rename("revenue(천원)").to_frame()
    tail.index = tail.index.astype(str)  # '2024Q1' 형태

    # 모델별 예측
    fc, raw = _run_models(series, forecast_quarters)
    fc_display = fc.copy()
    fc_display.index = fc_display.index.astype(str)  # 예측 분기 '2025Q1' 형태

    if show:
        print("=" * 70)
        print(f" 종목: {ticker_str}   |   조회 {n_raw}행 -> 사용 {n_obs}분기"
              f"   |   예측 {forecast_quarters}분기   |   앙상블: {ENSEMBLE_AGG}")
        print("=" * 70)
        print(f"\n[ 예측 입력 매출 — 최근 {min(tail_n, n_obs)}개 분기 (tail) ]")
        print(tail.to_string(float_format=lambda x: f"{x:,.0f}"))

        # 모델별 적합 상태 (v4: spec 경로 수정 — v3는 항상 None 이었음)
        print("\n[ 모델 적합 상태 ]")
        for nm in ("SARIMA", "ETS", "Theta"):
            r = raw.get(nm, {}) or {}
            if "forecast" in r:
                spec = r.get("spec", {})
                extra = ""
                if nm == "SARIMA":
                    extra = f"  order={spec.get('order')} seasonal={spec.get('seasonal_order')}"
                tf = ",".join(r.get("used_transform", [])) or "none"
                print(f"   {nm:<7} 성공   transform={tf}{extra}")
            else:
                print(f"   {nm:<7} 실패   {r.get('error', 'unknown')}")

        print(f"\n[ 모델별 예측값 ]")
        print(fc_display.to_string(float_format=lambda x: f"{x:,.0f}"))
        print("=" * 70)

    return {
        "ticker": ticker_str,
        "n_obs": n_obs,
        "input_series": series,
        "input_tail": tail,
        "forecast": fc,
        "raw_results": raw,
    }

## 5. Long 변환 · 배치 저장 함수

In [10]:
def convert_to_long_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Wide(date,ticker,SARIMA,ETS,Theta,Ensemble) → Long(date,ticker,indicator,value)
    """
    if df.empty:
        return pd.DataFrame(columns=['date', 'ticker', 'indicator', 'value'])

    # n_models 는 메타정보이므로 melt 대상에서 제외 (v4)
    model_cols = ['SARIMA', 'ETS', 'Theta', 'Ensemble']
    long_df = df.melt(
        id_vars=['date', 'ticker'],
        value_vars=model_cols,
        var_name='indicator',
        value_name='value'
    )
    return long_df.sort_values(['ticker', 'date', 'indicator']).reset_index(drop=True)


VINTAGE_KEY_COLS = "date,ticker,indicator,created_at"
LEGACY_KEY_COLS = "date,ticker,indicator"


def ensure_vintage_key(cursor, table_name: str) -> str:
    """
    실제 DB 테이블의 UNIQUE KEY 가 vintage 4열 키인지 검사하고, 아니면 교정한다 (멱등).

    v4까지의 문제: 코드의 CREATE TABLE IF NOT EXISTS 에는 4열 키가 있었지만,
    테이블이 이미 존재하면 이 문장은 아무것도 하지 않는다. 구버전이 만든
    (date, ticker, indicator) 3열 키가 살아 있으면 재실행 때마다 과거 예측이
    덮어써져 vintage 가 소멸한다. 매 저장 전에 여기서 보증한다.

    Returns: 'ok' | 'altered' | 'created'
    """
    cursor.execute(
        """
        SELECT INDEX_NAME, GROUP_CONCAT(COLUMN_NAME ORDER BY SEQ_IN_INDEX) AS cols
        FROM information_schema.STATISTICS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = %s
          AND NON_UNIQUE = 0
          AND INDEX_NAME <> 'PRIMARY'
        GROUP BY INDEX_NAME
        """,
        (table_name,),
    )
    uniques = {name: cols for name, cols in cursor.fetchall()}

    if VINTAGE_KEY_COLS in uniques.values():
        return "ok"

    legacy = [n for n, c in uniques.items() if c == LEGACY_KEY_COLS]
    if legacy:
        # 3열 키가 unique 였다면 4열 조합은 자동으로 unique 이므로 ALTER 는 안전하다.
        cursor.execute(
            f"ALTER TABLE {table_name} "
            f"DROP INDEX `{legacy[0]}`, "
            f"ADD UNIQUE KEY unique_date_ticker_indicator_created "
            f"(date, ticker, indicator, created_at)"
        )
        print(f"  [v5] vintage 키 교정: `{legacy[0]}`({LEGACY_KEY_COLS}) -> (date,ticker,indicator,created_at)")
        return "altered"

    # unique 키가 아예 없던 경우 — 중복이 있으면 여기서 실패하며, 그때는 §6 감사로 원인 확인
    cursor.execute(
        f"ALTER TABLE {table_name} "
        f"ADD UNIQUE KEY unique_date_ticker_indicator_created "
        f"(date, ticker, indicator, created_at)"
    )
    print("  [v5] vintage 키 신규 추가: (date,ticker,indicator,created_at)")
    return "created"


def save_forecasts_batch(
        forecasts_long: pd.DataFrame,
        db_info: dict,
        table_name: str = "korea_revenue_forecast_result",
        batch_size: int = 50
) -> Tuple[int, int, int]:
    """
    예측 결과(Long)를 배치로 DB에 저장. created_at(실행 날짜)으로 버전 구분.
    UNIQUE KEY (date, ticker, indicator, created_at) → 같은 날 재실행 시 value 갱신,
    다른 날 실행분은 별도 행으로 보존 (vintage). v5: 키 구성을 매 실행 시 검증·교정.

    Returns: (신규 삽입 수, 업데이트 수, 총 시도 수)
    """
    if forecasts_long.empty:
        print("저장할 데이터가 없습니다")
        return 0, 0, 0

    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"], user=db_info["user"],
        password=db_info["password"], db=db_info["database"],
        charset="utf8mb4", autocommit=False,
    )
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INT AUTO_INCREMENT PRIMARY KEY,
            date DATE NOT NULL COMMENT '예측 대상 날짜',
            ticker VARCHAR(20) NOT NULL,
            indicator VARCHAR(50) NOT NULL,
            value DOUBLE,
            created_at DATE NOT NULL COMMENT '예측 실행 날짜 (DATE만)',
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            UNIQUE KEY unique_date_ticker_indicator_created (date, ticker, indicator, created_at),
            INDEX idx_ticker (ticker),
            INDEX idx_date (date),
            INDEX idx_indicator (indicator),
            INDEX idx_created_at (created_at)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
        COMMENT='한국 기업 매출 예측 결과 (날짜별 버전 관리)'
        """)

        # v5: 기존 테이블이어도 vintage 키를 실제로 보증 (CREATE IF NOT EXISTS 는 기존 테이블에 무력)
        ensure_vintage_key(cursor, table_name)
        conn.commit()

        today_date = datetime.now().date()
        insert_sql = f"""
        INSERT INTO {table_name} (date, ticker, indicator, value, created_at)
        VALUES (%s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value), updated_at = CURRENT_TIMESTAMP
        """

        total_new = total_updated = total_attempted = 0
        total_rows = len(forecasts_long)

        for i in range(0, total_rows, batch_size):
            batch_df = forecasts_long.iloc[i:i + batch_size]
            batch_data = [
                (row['date'], row['ticker'], row['indicator'],
                 float(row['value']) if pd.notna(row['value']) else None, today_date)
                for _, row in batch_df.iterrows()
            ]
            cursor.executemany(insert_sql, batch_data)
            affected = cursor.rowcount

            if affected == len(batch_data):
                total_new += len(batch_data)
            elif affected == len(batch_data) * 2:
                total_updated += len(batch_data)
            else:
                updated = max(0, affected - len(batch_data))
                total_new += len(batch_data) - updated
                total_updated += updated
            total_attempted += len(batch_data)
            conn.commit()

        print(f"  저장 완료 | 시도 {total_attempted:,} · 신규 {total_new:,} · 갱신 {total_updated:,} · 실행일 {today_date}")
        return total_new, total_updated, total_attempted

    except Exception as e:
        conn.rollback()
        print(f"DB 저장 실패: {e}")
        raise
    finally:
        conn.close()

## 6. 일괄 예측 실행 함수 — `process_all_tickers()`

예측 결과를 `BATCH_SIZE`개씩 버퍼에 모았다가 Long 변환 후 한 번에 저장합니다.
`GC_EVERY`개마다 `gc.collect()`로 모델 적합 잔여 메모리를 정리합니다.

In [11]:
def process_all_tickers(
        tickers: List[str],
        db_info: dict,
        forecast_quarters: int = 8,
        min_data_periods: int = 24,
        batch_size: int = 50,
        gc_every: int = 10,
        verbose: bool = True
) -> Dict:
    """
    다수 ticker 예측 + 배치 저장.
    - batch_size: 버퍼에 모았다 한 번에 저장할 ticker 수
    - gc_every: 몇 ticker마다 gc.collect()를 호출할지
    """
    total = len(tickers)
    success_count = fail_count = 0
    failed_tickers = []
    batch_buffer = []

    print("=" * 80)
    print(f"예측 시작 | 대상 {total}개 · 예측 {forecast_quarters}분기 · 최소 {min_data_periods}분기 · 배치 {batch_size}")
    print("=" * 80)
    start_time = datetime.now()

    def _flush(buffer):
        if not buffer:
            return 0
        combined_long = convert_to_long_format(pd.concat(buffer, ignore_index=True))
        new, _, _ = save_forecasts_batch(combined_long, db_info, batch_size=batch_size)
        return new

    for i, ticker in enumerate(tqdm(tickers, desc="예측 진행"), 1):
        success, result_df, error_msg = forecast_quarterly_revenue(
            ticker=ticker, db_info=db_info,
            forecast_quarters=forecast_quarters,
            min_data_periods=min_data_periods, verbose=False,
        )
        if success:
            success_count += 1
            batch_buffer.append(result_df)
            if len(batch_buffer) >= batch_size:
                _flush(batch_buffer)
                batch_buffer = []
        else:
            fail_count += 1
            failed_tickers.append((ticker, error_msg))

        # 주기적 메모리 정리
        if gc_every and (i % gc_every == 0):
            gc.collect()

    # 남은 버퍼 저장
    _flush(batch_buffer)
    gc.collect()

    elapsed = (datetime.now() - start_time).total_seconds()

    print("\n" + "=" * 80)
    print("예측 완료")
    print("=" * 80)
    print(f"총 처리 시간: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
    print(f"성공: {success_count}개 ({success_count/total*100:.1f}%)")
    print(f"실패: {fail_count}개 ({fail_count/total*100:.1f}%)")
    if total:
        print(f"평균 처리 시간: {elapsed/total:.2f}초/ticker")

    if failed_tickers:
        error_summary = {}
        for _, error in failed_tickers:
            error_summary[error] = error_summary.get(error, 0) + 1
        print(f"\n실패 원인 요약:")
        for error, count in sorted(error_summary.items(), key=lambda x: -x[1]):
            print(f"  {error}: {count}개")

        # v4: 결측분기로 인한 탈락이 많으면 원천 데이터 점검 필요
        n_short = sum(c for e, c in error_summary.items() if '데이터 부족' in e)
        if n_short and n_short / max(fail_count, 1) > 0.5:
            print(f"\n  [진단] 실패의 {n_short/fail_count:.0%}가 '데이터 부족'입니다.")
            print("         결측분기로 연속구간이 짧아졌을 수 있습니다.")
            print("         inspect_ticker(문제티커) 로 조회행수 대비 사용분기를 확인하세요.")
        sample = failed_tickers if len(failed_tickers) <= 20 else failed_tickers[:20]
        print(f"\n실패 ticker {'목록' if len(failed_tickers) <= 20 else '샘플(20개)'}:")
        for t, e in sample:
            print(f"  {t}: {e}")

    return {'total': total, 'success': success_count, 'fail': fail_count,
            'failed_tickers': failed_tickers, 'elapsed_time': elapsed}

## 7. 실행

### 7-1. 공통 설정 · DB 연결

In [12]:
# ===== 예측 설정 =====
FORECAST_QUARTERS = 8     # 예측할 분기 수
MIN_DATA_PERIODS  = 24    # 최소 데이터 기간 (24분기 = 6년). 연속구간 추출 '이후' 기준
BATCH_SIZE        = 50    # 배치 저장 크기
GC_EVERY          = 10    # 몇 ticker마다 gc.collect()

# 앙상블 집계 방식 — 'median'(기본, 이상치 방어) / 'mean'(v3 동작)
ENSEMBLE_AGG = "mean"

db_info = get_db_info()
print("DB 연결 정보 준비 완료")
print(f"  예측 {FORECAST_QUARTERS}분기 · 최소 {MIN_DATA_PERIODS}분기 · 앙상블 {ENSEMBLE_AGG}")
print(f"  저장 표준: ticker={'A005930형' if RESULT_TICKER_HAS_A_PREFIX else '005930형'} · 단위={RESULT_VALUE_UNIT} (환산계수 {RESULT_UNIT_FACTOR})")


DB 연결 정보 준비 완료
  예측 8분기 · 최소 24분기 · 앙상블 mean
  저장 표준: ticker=005930형 · 단위=천원 (환산계수 1.0)


### 7-2. 🔍 개별 종목 점검 (DB 저장 없음)

예측 타당성을 확인하려는 종목 코드를 넣고 실행하세요. 입력 매출 `tail(20)`과 모델별 예측값이 표시됩니다.

In [13]:
# 점검할 종목 코드 (예: 삼성전자)
INSPECT_TICKER = "003350"

result = inspect_ticker(
    ticker=INSPECT_TICKER,
    db_info=db_info,
    forecast_quarters=FORECAST_QUARTERS,
    tail_n=20,
)

# 반환 객체로 추가 분석 가능
# result["input_tail"]   # 입력 매출 최근 20분기
# result["forecast"]     # 모델별 예측값 (wide)

[메모리] forecast_sarima 실행 전: 430.00 MB
[메모리] find_best_sarima_params 실행 전: 430.01 MB
[메모리] find_best_sarima_params 실행 후: 435.63 MB (변화: +5.62 MB)
[메모리] forecast_sarima 실행 후: 435.71 MB (변화: +5.71 MB)
[메모리] forecast_ets 실행 전: 435.71 MB
[메모리] forecast_ets 실행 후: 436.02 MB (변화: +0.30 MB)
[메모리] forecast_theta 실행 전: 436.02 MB
[메모리] forecast_theta 실행 후: 436.19 MB (변화: +0.18 MB)
 종목: A003350   |   조회 67행 -> 사용 67분기   |   예측 8분기   |   앙상블: mean

[ 예측 입력 매출 — 최근 20개 분기 (tail) ]
        revenue(천원)
period             
2021Q3   15,937,713
2021Q4   15,058,687
2022Q1   28,012,787
2022Q2   31,513,623
2022Q3   24,024,481
2022Q4   20,326,078
2023Q1   26,908,179
2023Q2   27,771,951
2023Q3   28,171,549
2023Q4   26,486,083
2024Q1   35,135,117
2024Q2   48,114,103
2024Q3   48,222,880
2024Q4   35,997,122
2025Q1   37,998,331
2025Q2   53,773,985
2025Q3   54,836,233
2025Q4   38,019,754
2026Q1   52,227,149
2026Q2   71,182,564

[ 모델 적합 상태 ]
   SARIMA  성공   transform=log1p  order=(1, 0, 2) seasonal=(1, 0, 0, 4)
   E

### 7-3. 일괄 예측 + DB 저장

아래 셀에서 대상 종목을 정한 뒤 실행합니다.
- **테스트:** `RUN_TICKERS = all_tickers[:30]`
- **특정 종목만:** `RUN_TICKERS = ["005930", "000660"]`
- **구간 지정:** `all_tickers[TICKER_START:TICKER_END]`
- **전체:** `RUN_TICKERS = all_tickers`

In [14]:
# 전체 ticker 목록 조회
all_tickers = get_all_tickers(db_info)

# ----- 실행 대상 선택 (원하는 줄의 주석을 해제) -----
# RUN_TICKERS = all_tickers[:30]        # 테스트: 앞 30개
# RUN_TICKERS = ["005930", "000660"]  # 특정 종목만
# TICKER_START, TICKER_END = 0, 500
# RUN_TICKERS = all_tickers[TICKER_START:TICKER_END]   # 구간 지정
# RUN_TICKERS = all_tickers           # 전체

RUN_TICKERS =  ["A003350"]

print(f"실행 대상: {len(RUN_TICKERS)}개  (샘플: {RUN_TICKERS[:5]})")

매출 데이터 보유 ticker 1582개 조회 완료 (소스: korea_fs_data_from_DG)

실행 대상: 1개  (샘플: ['A003350'])


In [15]:
result = process_all_tickers(
    tickers=RUN_TICKERS,
    db_info=db_info,
    forecast_quarters=FORECAST_QUARTERS,
    min_data_periods=MIN_DATA_PERIODS,
    batch_size=BATCH_SIZE,
    gc_every=GC_EVERY,
    verbose=True,
)

예측 시작 | 대상 1개 · 예측 8분기 · 최소 24분기 · 배치 50


예측 진행:   0%|          | 0/1 [00:00<?, ?it/s]

[메모리] forecast_sarima 실행 전: 436.36 MB
[메모리] find_best_sarima_params 실행 전: 436.36 MB
[메모리] find_best_sarima_params 실행 후: 437.57 MB (변화: +1.20 MB)
[메모리] forecast_sarima 실행 후: 437.57 MB (변화: +1.20 MB)
[메모리] forecast_ets 실행 전: 437.57 MB
[메모리] forecast_ets 실행 후: 437.57 MB (변화: +0.00 MB)
[메모리] forecast_theta 실행 전: 437.57 MB
[메모리] forecast_theta 실행 후: 437.57 MB (변화: +0.00 MB)
  저장 완료 | 시도 32 · 신규 32 · 갱신 0 · 실행일 2026-09-25

예측 완료
총 처리 시간: 8.4초 (0.1분)
성공: 1개 (100.0%)
실패: 0개 (0.0%)
평균 처리 시간: 8.38초/ticker


In [16]:
pd.Series(result)


total                    1
success                  1
fail                     0
failed_tickers          []
elapsed_time      8.382159
dtype: object

In [24]:
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4".format(
        user=db_info["user"], password=db_info["password"],
        host=db_info["host"], port=db_info.get("port", 3306),
        database=db_info["database"],
    )
)

# ── 1) 최신 버전(created_at 기준) 예측 조회 — ticker 두 형식 모두 커버 ──
sql = text("""
    SELECT date, ticker, indicator, value, created_at
    FROM korea_revenue_forecast_result
    WHERE ticker IN (:t1, :t2)
      AND created_at = (
          SELECT MAX(created_at)
          FROM korea_revenue_forecast_result
          WHERE ticker IN (:t1, :t2)
      )
    ORDER BY indicator, date
""")
fc = pd.read_sql(sql, engine, params={"t1": "A003350", "t2": "003350"})
fc["date"] = pd.to_datetime(fc["date"])
print(f"버전(created_at): {fc['created_at'].max()} | ticker: {list(fc['ticker'].unique())} | {len(fc)}행")

# ── 2) 분기 × 모델 피벗 (천원 → 억원) ──
pivot = (fc.assign(quarter=fc["date"].dt.to_period("Q").astype(str),
                   value_억원=fc["value"] / 1e5)
           .pivot_table(index="quarter", columns="indicator", values="value_억원"))
display(pivot.round(1))

버전(created_at): 2026-09-25 00:00:00 | ticker: ['003350'] | 32행


indicator,ETS,Ensemble,SARIMA,Theta
quarter,,,,
2026Q3,521.0,563.4,658.9,510.3
2026Q4,508.2,523.5,569.3,493.1
2027Q1,608.1,617.7,654.1,590.8
2027Q2,736.0,733.8,749.6,715.6
2027Q3,571.0,621.0,751.4,540.6
2027Q4,557.0,604.9,735.5,522.3
2028Q1,666.5,695.8,794.7,626.1
2028Q2,806.8,807.8,857.9,758.8


In [23]:
fc

,date,ticker,indicator,value,created_at
0,2026-09-30,003350,Ensemble,5.633891e+07,2026-09-25
1,2026-12-31,003350,Ensemble,5.235198e+07,2026-09-25
2,2027-03-31,003350,Ensemble,6.176524e+07,2026-09-25
3,2027-06-30,003350,Ensemble,7.337510e+07,2026-09-25
4,2027-09-30,003350,Ensemble,6.210138e+07,2026-09-25
5,2027-12-31,003350,Ensemble,6.049369e+07,2026-09-25
6,2028-03-31,003350,Ensemble,6.957527e+07,2026-09-25
7,2028-06-30,003350,Ensemble,8.078197e+07,2026-09-25
8,2026-09-30,003350,ETS,5.209598e+07,2026-09-25
9,2026-12-31,003350,ETS,5.082019e+07,2026-09-25


### 7-4. 실패 종목 재시도 (선택)

In [ ]:
if result['failed_tickers']:
    retry_tickers = [t[0] for t in result['failed_tickers']]
    print(f"재시도 대상: {len(retry_tickers)}개\n")
    retry_result = process_all_tickers(
        tickers=retry_tickers,
        db_info=db_info,
        forecast_quarters=FORECAST_QUARTERS,
        min_data_periods=MIN_DATA_PERIODS,
        batch_size=BATCH_SIZE,
        gc_every=GC_EVERY,
        verbose=True,
    )
else:
    print("실패한 ticker가 없습니다.")